# 07 — Secondary full clinical network with derived indices (sensitivity only)

Complementary network that adds the deterministic derived indices (HOMA-IR, FAI, TyG, NLR, etc.) back in, using unconstrained min-EBIC selection. Reported strictly as a sensitivity analysis (Methods 2.4, 2.11) — never used for the primary biological interpretation.

## Environment setup

In [ ]:
import os, sys, subprocess
from pathlib import Path


def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


PROJECT_ROOT = Path.cwd().resolve()
while not ((PROJECT_ROOT / "scripts").exists() and (PROJECT_ROOT / "outputs").exists()):
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

if _in_colab():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                     "scikit-learn", "networkx", "statsmodels", "openpyxl"], check=True)

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
from scripts._pipeline_common import *  # noqa: F401,F403

print("Project root:", PROJECT_ROOT)


In [ ]:
import json
import numpy as np, pandas as pd

fd = pd.read_csv(PROJECT_ROOT / 'outputs' / '01_ingest_harmonize' / 'feature_dictionary_v2.csv')
secondary = fd.loc[fd.included_secondary_full_clinical, 'feature'].tolist()
all_dom = {**DOM, **DERIVED}
doms = {f: all_dom[f] for f in secondary}

H = pd.read_csv(PROJECT_ROOT / 'outputs' / '01_ingest_harmonize' / 'harmonized_features.csv')
Xs = H[secondary]
Ys, Ysi, Zs, Ts = preprocess(Xs)

ms, alpha_table_s, selected_alpha_s = select_glasso_ebic(Zs)

out_dir = PROJECT_ROOT / 'outputs' / '07_secondary_full_clinical_network'
out_dir.mkdir(parents=True, exist_ok=True)
alpha_table_s.to_csv(out_dir / 'secondary_alpha_selection_ebic.csv', index=False)

pr = ms.precision_
d = np.sqrt(np.diag(pr))
Ps = -pr / np.outer(d, d)
np.fill_diagonal(Ps, 1)
es = edge_df_from_partial(Ps, secondary)
es['source_domain'] = es.source.map(doms)
es['target_domain'] = es.target.map(doms)
es['cross_domain'] = es.source_domain != es.target_domain
es.to_csv(out_dir / 'secondary_full_clinical_edges_with_derived.csv', index=False)

mets, _ = graph_metrics(es, {f: doms[f] for f in secondary})
mets.to_csv(out_dir / 'secondary_centrality_bridge_metrics_with_derived.csv', index=False)
mets.head(10).to_csv(out_dir / 'top_bridge_biomarkers_secondary_with_derived.csv', index=False)

(out_dir / 'secondary_model_summary.json').write_text(json.dumps({
    'n_features': len(secondary), 'selected_alpha_ebic': float(selected_alpha_s),
    'n_edges': int(len(es)), 'network_density': float(len(es) / (len(secondary) * (len(secondary) - 1) / 2)),
    'note': 'Sensitivity analysis only: includes mathematically derived ratios/indices excluded from primary network.',
}, indent=2), encoding='utf-8')

print(f'Secondary features: {len(secondary)}; edges: {len(es)}')
mets.head(10)
